# 網頁運作原理

## GET

### [練習] 找尋 API 網址
但實際上我們所需要的並不是圖本身，而是圖上的數字，  
因為我們需要的是「每日股價」，請先將股價圖切換至「全部」，    
再利用開發人員工具搜尋圖上的數字，來找到資料對應的網址

> 不建議拿最新日期數字來查詢，因為重複出現機率滿高的

```
https://tw.stock.yahoo.com/_____/___/resource/FinanceChartService.ApacLibraCharts;...symbols=%5B%22NVDA%22%5D...
```

In [ ]:
# Ans
# https://tw.stock.yahoo.com/_td-stock/api/resource/FinanceChartService.ApacLibraCharts;autoRefresh=1771418747696;period=1d;range=6mo;symbols=%5B%22NVDA%22%5D;type=null?bkt=c1-stock-lumos&device=desktop&ecma=modern&feature=enableGAMAds%2CenableGAMEdgeToEdge%2CenableEvPlayer%2CuseCG%2CuseCGV2&intl=tw&lang=zh-Hant-TW&partner=none&prid=2bp3ug5kpbd39&region=TW&site=finance&tz=Asia%2FTaipei&ver=1.4.825&returnMeta=true
# https://tw.stock.yahoo.com/_td-stock/api/resource/FinanceChartService.ApacLibraCharts;symbols=%5B%22NVDA%22%5D

# Crawling

## [stooq](https://stooq.com/)

### 抓取網頁內容

#### [練習] 可抓取多頁的爬蟲
如果我們要抓取所有每日歷史資料，只抓第一頁是不夠的，  
可以利用 for 迴圈，配合字串格式化替換網址頁數，就可以抓取多頁的網頁內容。

##### 實戰要點
- 很多頁的資料，可以建立一個專屬的資料夾存放。
- 需要設定兩次間隔抓取的時間，降低被網站封鎖的風險

In [ ]:
import requests
import os
import random
import time


dir_name = "nvda_stooq"

### 建立資料夾
if dir_name not in os.listdir():
    os.makedirs(dir_name)

for page in range(1,4): # will crawling page 1,2,3
    url = f"https://stooq.com/q/d/?s=nvda.us&i=d&l={page}"
    response = requests.get(url)

    with open(f"{dir_name}/{page}.html","w") as fo:
        fo.write(response.text)
    time.sleep(random.randint(3,7)) # 程式暫停 3~7秒

In [ ]:
# 檢查網頁上想要的資訊是否有在網頁原始碼中
with open("nvda_stooq/1.html") as fi:
    html = fi.read()
    print("某日價格" in html)

## [NASDAQ](https://www.nasdaq.com/)

### [練習] 找尋 API 網址
如果要抓所有的歷史資料，要記得將圖表切到「MAX」，  
開啟開發人員工具的 Network 分頁，然後重新整理網頁，  
再利用開發人員工具搜尋圖上的數字，來找到資料對應的網址

> - 建議拿小數點後有三碼的數字，重複率較低
> - 不建議拿最新日期數字來查詢，因為重複出現機率滿高的
> - 不建議拿成交量，因為千位數逗號不見得是原始資料

https://www.nasdaq.com/market-activity/stocks/nvda/historical?page=1&rows_per_page=10&timeline=y10

In [ ]:
import requests
url = "https://api.nasdaq.com/api/quote/NVDA/historical?assetclass=stocks&fromdate=2016-03-21&limit=10000&todate=2026-03-21"
headers = {"user-agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"}
response = requests.get(url, headers=headers)

with open("nvda_nasdaq_api.html","w") as fo:
    fo.write(response.text)

# Parsing

## [JSON](https://docs.python.org/zh-tw/3/library/json.html)

### [練習] 解析 JSON

In [ ]:
from datetime import datetime
import json

with open("nvda_nasdaq_api.html") as fi:
    html = fi.read()
    print(html)

json_data = json.loads(html)
dates = []
opens = []
highs = []
lows = []
closes = []
volumes = []
for row in json_data["data"]["tradesTable"]["rows"]:
    date = row["date"]
    date_ymd = datetime.strptime(date, "%m/%d/%Y").strftime("%Y-%m-%d")
    dates.append(date_ymd)
    opens.append(row["open"].replace("$",""))
    highs.append(row["high"].replace("$",""))
    lows.append(row["low"].replace("$",""))
    closes.append(row["close"].replace("$",""))
    volumes.append(row["volume"])

data = list(zip(dates, opens, highs, lows, closes, volumes))
print(data[0])

# 整合實作

## [練習] Crawling 函式
請將原來的程式碼改寫成函式
``` python
import requests
url = "___________" # 之前 NASDAQ 練習題的答案
headers = {"user-agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"}
response = requests.get(url, headers=headers)

with open("nvda_nasdaq_api.html","w") as fo:
    fo.write(response.text)
```

In [ ]:
import requests
FILENAME_SUFFIX = "_nasdaq_api"
def crawling(stock_id):
    url = f"https://api.nasdaq.com/api/quote/{stock_id}/historical?assetclass=stocks&fromdate=2016-03-21&limit=10000&todate=2026-03-21" # 此題練習的地方 + 之前 NASDAQ 練習題的答案
    headers = {"user-agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"}
    response = requests.get(url, headers=headers)
    with open(f"{stock_id}{FILENAME_SUFFIX}.html","w") as fo: # 此題練習的地方
        fo.write(response.text)

crawling("NVDA")
crawling("GOOG")
crawling("AAPL")

## [練習] Parsing 函式
請將原來的程式碼改寫成函式
``` python
from datetime import datetime
import json

with open("nvda_nasdaq_api.html") as fi:
    html = fi.read()

json_data = json.loads(html)
dates = []
opens = []
highs = []
lows = []
closes = []
volumes = []
for row in json_data["_____"]["_______"]["rows"]: # 之前練習題的答案
    date = row["date"]
    date_ymd = datetime.strptime(date, "%m/%d/%Y").strftime("%Y-%m-%d")
    dates.append(date_ymd)
    opens.append(row["open"].replace("$",""))
    highs.append(row["high"].replace("$",""))
    lows.append(row["low"].replace("$",""))
    closes.append(row["close"].replace("$",""))
    volumes.append(row["volume"])
    
data = list(zip(dates, opens, highs, lows, closes, volumes))
print(data[0])
```

In [ ]:
from datetime import datetime
import json

FILENAME_SUFFIX = "_nasdaq_api"

def parsing(stock_id): # 此題練習的地方
    with open(f"{stock_id}{FILENAME_SUFFIX}.html") as fi:  # 此題練習的地方
        html = fi.read()

    json_data = json.loads(html)
    dates = []
    opens = []
    highs = []
    lows = []
    closes = []
    volumes = []
    for row in json_data["data"]["tradesTable"]["rows"]: # 之前練習題的答案
        date = row["date"]
        date_ymd = datetime.strptime(date, "%m/%d/%Y").strftime("%Y-%m-%d")
        dates.append(date_ymd)
        opens.append(row["open"].replace("$",""))
        highs.append(row["high"].replace("$",""))
        lows.append(row["low"].replace("$",""))
        closes.append(row["close"].replace("$",""))
        volumes.append(row["volume"])

    data = list(zip(dates, opens, highs, lows, closes, volumes))
    print(data[0])

parsing("NVDA")
parsing("GOOG")
parsing("AAPL")

## 合併所有函式

In [ ]:
import csv
import json
import requests
import time

FILENAME_SUFFIX = "_nasdaq_api"

def save_csv(data, filename):
    with open(filename, "w") as fo:
        writer = csv.writer(fo)
        writer.writerows(data)

def crawling(stock_id):
    url = f"https://api.nasdaq.com/api/quote/{stock_id}/historical?assetclass=stocks&fromdate=2016-03-21&limit=10000&todate=2026-03-21"
    headers = {"user-agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"}
    response = requests.get(url, headers=headers)
    with open(f"{stock_id}{FILENAME_SUFFIX}.html","w") as fo:
        fo.write(response.text)

def parsing(stock_id):
    with open(f"{stock_id}{FILENAME_SUFFIX}.html") as fi:
        html = fi.read()

    json_data = json.loads(html)
    dates = []
    opens = []
    highs = []
    lows = []
    closes = []
    volumes = []
    for row in json_data["data"]["tradesTable"]["rows"]:
        date = row["date"]
        date_ymd = datetime.strptime(date, "%m/%d/%Y").strftime("%Y-%m-%d")
        dates.append(date_ymd)
        opens.append(row["open"].replace("$",""))
        highs.append(row["high"].replace("$",""))
        lows.append(row["low"].replace("$",""))
        closes.append(row["close"].replace("$",""))
        volumes.append(row["volume"])

    data = list(zip(dates, opens, highs, lows, closes, volumes))
    print(data[0])

    csv_header = ["Date","Open","High","Low","Close","Volume"]
    data  = [csv_header] + data
    save_csv(data, f"{stock_id}{FILENAME_SUFFIX}.csv")


stock_ids = ["NVDA", "GOOG", "AAPL"]

for stock_id in stock_ids:
    crawling(stock_id)
    time.sleep(random.randint(3,7)) # 程式暫停 3~7秒

for stock_id in stock_ids:
    parsing(stock_id)